# 🧠 Sesión 9 · Reconocer números escritos a mano

Hoy la red va a **ver imágenes**: números del 0 al 9 escritos a mano por personas reales.

Es el conjunto de datos **MNIST**: 70 000 imágenes. Viene incluido en Keras.

⚡ **Antes de empezar:** en Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**. Así la red entrena mucho más rápido.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
import mlcroissant as mlc
import pandas as pd

## 1. Cargar las imágenes

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("Imágenes para aprender:", len(X_train))
print("Imágenes para el examen:", len(X_test))

In [ ]:
# Cargar los datos desde un CSV local
filename='https://www.kaggle.com/datasets/sachinpatel21/az-handwritten-alphabets-in-csv-format'
data = pd.read_csv("./csv/A_Z Handwritten Data.csv").astype("float32")

# Separar características (píxeles) y etiquetas
X = data.drop("0", axis=1).values
y = data["0"].values

# Redimensionar para un modelo de Keras (28x28 píxeles, 1 canal)
X = X.reshape(-1, 28, 28)

data.info()

In [ ]:
print(f"imprimir las columnas de las imagenes X_train {X_train.shape}")
print(f"imprimir las columnas de las imagenes X {X.shape}")
print(f"imprimir las columnas de las imagenes X {X[:10]}")

## 2. Ver algunas imágenes

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Es un {y_train[i]}")
    plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X[i], cmap="gray")
    plt.title(f"Es un {y[i]}")
    plt.axis("off")
plt.show()

## 3. ¿Qué ve el ordenador?
Para el ordenador, una imagen es una **tabla de números**: 28 × 28 = **784 cuadraditos** (píxeles).
- **0** = negro
- **255** = blanco

In [ ]:
plt.imshow(X_train[0], cmap="gray")
plt.show()
print(X_train[0][10:15, 10:18])   # un trocito de la tabla de números

## 4. Preparar las imágenes
Dividimos entre 255 para que los números vayan de **0 a 1**. A la red le resulta más fácil.

In [ ]:
X_train = X_train / 255
X_test = X_test / 255

## 5. Crear la red
Lo nuevo:
- `keras.Input(shape=(28, 28))` → entra una imagen de 28 × 28
- `Flatten()` → pone los 784 cuadraditos **en fila**
- Salida `Dense(10, activation="softmax")` → **10 opciones** (del 0 al 9)

💡 Es la **plantilla C** (varias opciones) de la chuleta.

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(),                          # imagen → fila de 784 números
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")     # 10 números posibles
])
modelo.compile(loss="sparse_categorical_crossentropy",
               optimizer="adam",
               metrics=["accuracy"])

## 6. Entrenar
Ahora son **60 000 imágenes**, así que con **5 épocas** basta. Esta vez dejamos que escriba cómo va (`verbose=1`).

In [ ]:
modelo.fit(X_train, y_train, epochs=5, verbose=1)

## 7. El examen

In [ ]:
perdida, aciertos = modelo.evaluate(X_test, y_test, verbose=0)
print("Aciertos con 10 000 imágenes nuevas:", round(aciertos * 100, 1), "%")

## 8. Ver lo que dice la red

In [ ]:
predicciones = modelo.predict(X_test[:10], verbose=0)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i], cmap="gray")
    plt.title(f"La red dice: {np.argmax(predicciones[i])}")
    plt.axis("off")
plt.show()

## 9. ¿En qué se equivoca?
Buscamos imágenes donde la red ha fallado. A veces ¡ni nosotros sabríamos qué número es!

In [ ]:
todas = np.argmax(modelo.predict(X_test, verbose=0), axis=1)
fallos = np.where(todas != y_test)[0]
print("Ha fallado", len(fallos), "de", len(y_test))

plt.figure(figsize=(10, 4))
for i, n in enumerate(fallos[:10]):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[n], cmap="gray")
    plt.title(f"Era {y_test[n]}, dijo {todas[n]}", fontsize=9)
    plt.axis("off")
plt.show()

## 📝 Resumen
- Una imagen para el ordenador es una **tabla de números**.
- Dividimos entre **255** para que vayan de 0 a 1.
- `Flatten()` pone la imagen **en fila**.
- Es la plantilla de **varias opciones**: `Dense(10, activation="softmax")`.
- ¡Con 4 líneas de red acertamos más del **97 %**!